# Verifier Data And Training Validation

This notebook is the verifier data and training validation layer. It loads offline trace records, builds explicit verifier supervision from production trace artifacts, validates label consistency and balance, writes an inspectable verifier dataset, invokes the production verifier training pipeline, and surfaces calibration behavior without collapsing supervision to raw model confidence or hidden notebook heuristics.


In [ ]:
from __future__ import annotations

import json
import shutil
from collections import Counter
from pathlib import Path

import pandas as pd

from scripts import train_verifier as train_verifier_script
from src.common.metrics import canonical_match, evaluate_binary_calibration, stable_mean
from src.common.schemas import BranchTrace, FailureType, ReasoningStep
from src.offline.verifier_distillation import (
    ARTIFACT_KIND,
    DistillationSourceType,
    ExampleRetentionTag,
    SplitName,
    VerifierDistillationConfig,
    VerifierTrainingArtifact,
    VerifierTrainingArtifactMetadata,
    _NormalizedBundle,
    _branch_example,
    _prefix_example,
    _prefix_selections,
    _stable_hash,
    write_verifier_training_artifact,
)
from src.verifier.verifier_labels import bundle_from_branch_trace

SEED = 1337
OVERWRITE_OUTPUTS = True
MAX_BRANCH_LABEL_SHARE = 0.80
MIN_HARD_NEGATIVE_COUNT = 1
HARD_NEGATIVE_SYMBOLIC_FLOOR = 0.50

ROOT = Path.cwd()
INTERIM_TRACE_PATH = ROOT / 'data' / 'interim' / 'trace_records.parquet'
OFFLINE_TRACE_ROOT = ROOT / 'artifacts' / 'offline_trace_validation'
RUN_ID = f'verifier_training_validation_seed{SEED}'
OUT_DIR = ROOT / 'artifacts' / 'verifier_training_validation' / RUN_ID
VERIFIER_DATASET_PATH = OUT_DIR / 'verifier_dataset.parquet'
DIAGNOSTICS_PATH = OUT_DIR / 'verifier_diagnostics.json'
TRAINING_METRICS_PATH = OUT_DIR / 'training_metrics.json'
CALIBRATION_CURVES_PATH = OUT_DIR / 'calibration_curves.json'
ARTIFACT_ROOT = OUT_DIR / 'verifier_dataset_artifact'
TRAINING_ROOT = OUT_DIR / 'trainer_runs'

HARD_NEGATIVE_TAGS = {
    ExampleRetentionTag.PLAUSIBLE_WRONG,
    ExampleRetentionTag.SYMBOLIC_CONTRADICTED,
    ExampleRetentionTag.REPAIRED,
    ExampleRetentionTag.VERIFIER_REJECTED,
}


def need(condition, message):
    if not condition:
        raise AssertionError(message)


def sj(value):
    return json.dumps(value, ensure_ascii=True, sort_keys=True, default=str)


def text(value):
    return ' '.join(str(value or '').split())


def is_missing(value):
    return value is None or value != value


def parse_json(value, default):
    if is_missing(value):
        return default
    if isinstance(value, type(default)):
        return value
    if isinstance(value, (list, dict)):
        return value if isinstance(value, type(default)) else default
    raw = str(value).strip()
    if not raw:
        return default
    try:
        parsed = json.loads(raw)
    except Exception:
        return default
    return parsed if isinstance(parsed, type(default)) else default


def truthy(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return bool(value)
    return text(value).lower() in {'1', 'true', 'yes', 'y', 'pass', 'passed'}


def fnum(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return float(default)


def prepare_output_dir(path, overwrite):
    if path.exists():
        if not overwrite:
            raise FileExistsError(f'Output directory already exists: {path}')
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def resolve_trace_path():
    if INTERIM_TRACE_PATH.exists() and INTERIM_TRACE_PATH.stat().st_size > 0:
        return INTERIM_TRACE_PATH
    candidates = sorted(
        [path for path in OFFLINE_TRACE_ROOT.glob('**/trace_records.parquet') if path.stat().st_size > 0],
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    need(candidates, 'trace_records.parquet not found with non-zero size in data/interim or offline trace artifacts.')
    return candidates[0]


def route_snapshot_from_row(row):
    direct = parse_json(row.get('route_snapshot'), {})
    if direct:
        return direct
    provenance = parse_json(row.get('provenance'), {})
    route = provenance.get('route', {}) if isinstance(provenance, dict) else {}
    return route if isinstance(route, dict) else {}


def route_problem_type_from_snapshot(route_snapshot):
    payload = route_snapshot.get('problem_type_probs') or route_snapshot.get('problem_type') or {}
    return dict(payload) if isinstance(payload, dict) else {}


def route_archetypes_from_snapshot(route_snapshot):
    payload = route_snapshot.get('archetype_probs') or route_snapshot.get('archetypes') or {}
    return dict(payload) if isinstance(payload, dict) else {}


def step_objects_from_row(row):
    raw_steps = parse_json(row.get('step_sequence'), []) or parse_json(row.get('steps'), [])
    need(isinstance(raw_steps, list), 'Trace row steps must decode to a list.')
    steps = []
    for index, step in enumerate(raw_steps, start=1):
        need(isinstance(step, dict), f'Trace step {index} must be a mapping.')
        steps.append(
            ReasoningStep(
                step_num=int(fnum(step.get('step_num', step.get('step_index', index)), index)),
                description=text(step.get('description') or step.get('text')),
                operator_used=text(step.get('operator_used') or step.get('operator_name')) or None,
                symbolic_expression=text(step.get('symbolic_expression')) or None,
                symbolic_valid=truthy(step.get('symbolic_valid')) if 'symbolic_valid' in step else True,
                python_code=text(step.get('python_code')) or None,
                python_result=text(step.get('python_result')) or None,
            )
        )
    need(steps, 'Usable trace rows must contain at least one reasoning step.')
    return steps


def operator_sequence_from_row(row, steps):
    raw = parse_json(row.get('operator_sequence'), [])
    sequence = [text(item) for item in raw if text(item)] if isinstance(raw, list) else []
    if not sequence:
        sequence = [step.operator_used for step in steps if step.operator_used]
    return sequence


def failure_labels_from_row(row):
    labels = parse_json(row.get('failure_labels'), [])
    out = [text(label) for label in labels if text(label)] if isinstance(labels, list) else []
    for candidate in (row.get('failure_type'), row.get('failed_stage')):
        clean = text(candidate)
        if clean and clean not in out:
            out.append(clean)
    return out


def coerce_failure_type(value):
    clean = text(value)
    if not clean:
        return None
    for member in FailureType:
        if clean == member.value:
            return member
    return None


def trace_class_for_trace(trace, answer_correct):
    if trace.failure_type == FailureType.SYMBOLIC_MISMATCH:
        return 'symbolic_contradicted'
    if trace.repaired:
        return 'repaired_success' if answer_correct else 'repaired_failure'
    if answer_correct:
        return 'correct'
    return 'incorrect'


def is_hard_negative(example):
    if example.retention_tag in HARD_NEGATIVE_TAGS:
        return True, f'retention_tag:{example.retention_tag.value}'
    target = float(example.labels.model_targets.get('answer_correct_likelihood', 0.0))
    symbolic = float(example.labels.model_targets.get('symbolic_agreement', 0.0))
    if target < 0.5 and symbolic >= HARD_NEGATIVE_SYMBOLIC_FLOOR:
        return True, f'symbolic_floor:{HARD_NEGATIVE_SYMBOLIC_FLOOR:.2f}'
    return False, 'not_hard_negative'


def augment_example(example, case):
    hard_negative, reason = is_hard_negative(example)
    metadata = dict(example.metadata)
    metadata.update(
        {
            'hard_negative': bool(hard_negative),
            'hard_negative_reason': reason,
            'truth_source': case['truth_source'],
            'expected_answer': case['expected_answer'],
            'failure_labels': list(case['failure_labels']),
            'trace_status': case['trace_status'],
        }
    )
    return example.model_copy(update={'metadata': metadata})


def row_to_trace_case(row, index):
    record_type = text(row.get('record_type'))
    trace_status = text(row.get('trace_status'))
    problem_id = text(row.get('problem_id') or row.get('id'))
    branch_id = text(row.get('branch_id') or row.get('trace_id'))
    if record_type != 'trace_record' or trace_status == 'stage_failure' or not branch_id:
        return {
            'usable': False,
            'problem_id': problem_id or f'row_{index:05d}',
            'branch_id': branch_id or None,
            'exclude_reason': trace_status or record_type or 'missing_branch_id',
        }

    route_snapshot = route_snapshot_from_row(row)
    route_problem_type = route_problem_type_from_snapshot(route_snapshot)
    route_archetypes = route_archetypes_from_snapshot(route_snapshot)
    provenance = parse_json(row.get('provenance'), {})
    steps = step_objects_from_row(row)
    operator_sequence = operator_sequence_from_row(row, steps)
    step_operators = [step.operator_used for step in steps if step.operator_used]
    need(operator_sequence == step_operators, f'Operator alignment mismatch for {branch_id}.')
    failure_labels = failure_labels_from_row(row)
    expected_answer = text(row.get('expected_answer') or (provenance.get('expected_answer') if isinstance(provenance, dict) else None))
    need(expected_answer, f'Expected answer missing for trace {branch_id}.')
    answer = text(row.get('trace_answer') or row.get('answer')) or None
    answer_canonical = text(row.get('trace_answer_canonical') or row.get('answer_canonical') or answer) or None
    answer_correct = bool(answer_canonical) and canonical_match(expected_answer, answer_canonical)
    answer_likelihood = 1.0 if answer_correct else 0.0
    truth_source = 'expected_answer_exact_match'
    failure_type = coerce_failure_type(row.get('failure_type')) or coerce_failure_type(failure_labels[0] if failure_labels else None)
    trace = BranchTrace(
        branch_id=branch_id,
        problem_id=problem_id,
        steps=steps,
        full_reasoning='\n'.join(step.description for step in steps if step.description),
        answer=answer,
        answer_canonical=answer_canonical,
        failure_type=failure_type,
        failure_location=text(row.get('failure_location')) or None,
        symbolic_valid=truthy(row.get('symbolic_valid')) if 'symbolic_valid' in row else not failure_labels,
        branch_score=fnum(row.get('branch_score')),
        verifier_score=fnum(row.get('verifier_score')),
        tool_consistency=fnum(row.get('tool_consistency')),
        self_critiqued=truthy(row.get('self_critiqued')),
        repaired=truthy(row.get('repaired')),
        repair_count=int(fnum(row.get('repair_count'))),
        operator_sequence=operator_sequence,
        archetype_used=text(row.get('archetype_used')) or None,
        retrieval_used=truthy(row.get('retrieval_used')),
        generation_time_sec=fnum(row.get('generation_time_sec')),
    )
    bundle = bundle_from_branch_trace(
        trace,
        answer_correct=answer_correct,
        answer_correct_likelihood=answer_likelihood,
        route_problem_type=route_problem_type,
        route_archetypes=route_archetypes,
    )
    normalized = _NormalizedBundle(
        source_type=DistillationSourceType.BRANCH_TRACE,
        source_record_id=trace.branch_id,
        problem_id=trace.problem_id,
        branch_id=trace.branch_id,
        bundle=bundle,
        source_split=None,
        problem_text=text(row.get('problem_text')),
        reasoning_steps=tuple(
            {
                'description': step.description,
                'operator_used': step.operator_used,
                'symbolic_valid': bool(step.symbolic_valid),
                'symbolic_expression': step.symbolic_expression,
                'python_code': step.python_code,
                'python_result': step.python_result,
            }
            for step in trace.steps
        ),
        operator_sequence=tuple(trace.operator_sequence),
        answer_text=text(trace.answer),
        canonical_answer=text(trace.answer_canonical or trace.answer),
        metadata={
            'trace_class': trace_class_for_trace(trace, answer_correct),
            'trace_failure_type': trace.failure_type.value if trace.failure_type is not None else None,
            'trace_failure_location': trace.failure_location,
            'repaired': bool(trace.repaired),
            'repair_count': int(trace.repair_count),
            'retrieval_used': bool(trace.retrieval_used),
            'self_critiqued': bool(trace.self_critiqued),
            'expected_answer': expected_answer,
            'failure_labels': list(failure_labels),
            'truth_source': truth_source,
        },
    )
    return {
        'usable': True,
        'problem_id': problem_id,
        'branch_id': branch_id,
        'trace': trace,
        'bundle': bundle,
        'normalized': normalized,
        'answer_correct': answer_correct,
        'expected_answer': expected_answer,
        'truth_source': truth_source,
        'failure_labels': failure_labels,
        'trace_status': trace_status,
    }


def build_artifact_from_cases(cases, config):
    examples = []
    for case in cases:
        branch_example = augment_example(_branch_example(case['normalized'], config), case)
        examples.append(branch_example)
        for selection in _prefix_selections(case['normalized'].bundle, config):
            examples.append(augment_example(_prefix_example(case['normalized'], selection, config), case))

    need(examples, 'Verifier supervision examples were not constructed from trace records.')
    examples.sort(key=lambda item: (item.problem_id, item.branch_id, item.scope.value, item.prefix_step_count, item.example_id))

    split_counts = Counter(example.split.value for example in examples)
    scope_counts = Counter(example.scope.value for example in examples)
    retention_counts = Counter(example.retention_tag.value for example in examples)
    config_digest = _stable_hash('verifier_distill_config', config.model_dump(mode='json'))
    source_digest = _stable_hash(
        'verifier_distill_sources',
        [
            {
                'branch_id': case['branch_id'],
                'problem_id': case['problem_id'],
                'bundle_id': case['bundle'].bundle_id,
                'truth_source': case['truth_source'],
            }
            for case in cases
        ],
    )
    artifact_id = _stable_hash(
        'verifier_training_artifact',
        {
            'artifact_version': config.artifact_version,
            'config_digest': config_digest,
            'example_ids': [example.example_id for example in examples],
        },
    )

    artifact = VerifierTrainingArtifact(
        metadata=VerifierTrainingArtifactMetadata(
            artifact_kind=ARTIFACT_KIND,
            artifact_version=config.artifact_version,
            schema_version=config.schema_version,
            artifact_id=artifact_id,
            example_count=len(examples),
            source_count=len(cases),
            branch_count=len({case['branch_id'] for case in cases}),
            config_digest=config_digest,
            split_counts=dict(sorted(split_counts.items())),
            scope_counts=dict(sorted(scope_counts.items())),
            retention_counts=dict(sorted(retention_counts.items())),
            source_digest=source_digest,
        ),
        examples=tuple(examples),
    )
    return artifact


def answer_target(example):
    return 1 if float(example.labels.model_targets.get('answer_correct_likelihood', 0.0)) >= 0.5 else 0


def validate_artifact(artifact, excluded_rows):
    need(artifact.examples, 'Verifier artifact contains zero examples.')
    branch_examples = [example for example in artifact.examples if example.scope.value == 'branch']
    prefix_examples = [example for example in artifact.examples if example.scope.value == 'prefix']
    need(branch_examples, 'Verifier artifact must contain branch-scope examples.')
    need(prefix_examples, 'Verifier artifact must contain prefix-scope examples.')

    split_counts = Counter(example.split.value for example in artifact.examples)
    for split_name in (SplitName.TRAIN.value, SplitName.VALID.value, SplitName.CALIBRATION.value):
        need(split_counts.get(split_name, 0) > 0, f'Calibration-aware split `{split_name}` is empty.')

    hard_negative_count = 0
    for example in artifact.examples:
        targets = dict(example.labels.model_targets)
        step_scores = list(targets.get('step_correctness', ()))
        need(example.prefix_step_count <= example.total_step_count, f'Prefix steps exceed total steps for {example.example_id}.')
        need(example.total_step_count > 0, f'Total step count must be positive for {example.example_id}.')
        need(bool(example.texts.prefix_text or example.texts.reasoning_text), f'Reasoning text missing for {example.example_id}.')
        if example.scope.value == 'branch':
            need(len(step_scores) == example.total_step_count, f'Branch example step labels misaligned for {example.example_id}.')
        else:
            need(len(step_scores) == 1, f'Prefix example must expose exactly one step target for {example.example_id}.')
        if example.metadata.get('hard_negative'):
            hard_negative_count += 1

    need(hard_negative_count >= MIN_HARD_NEGATIVE_COUNT, 'Hard negative mining produced no explicit hard negatives.')

    branch_positive = [example for example in branch_examples if answer_target(example) == 1]
    branch_negative = [example for example in branch_examples if answer_target(example) == 0]
    need(branch_positive and branch_negative, 'Verifier dataset needs both correct and incorrect branch traces.')
    dominant_branch_share = max(len(branch_positive), len(branch_negative)) / len(branch_examples)
    need(dominant_branch_share <= MAX_BRANCH_LABEL_SHARE, f'Branch supervision is imbalanced: dominant_share={dominant_branch_share:.3f}.')

    positive_symbolic = [float(example.labels.model_targets.get('symbolic_agreement', 0.0)) for example in branch_positive]
    negative_symbolic = [float(example.labels.model_targets.get('symbolic_agreement', 0.0)) for example in branch_negative]
    need(stable_mean(positive_symbolic) >= stable_mean(negative_symbolic), 'Symbolic agreement is inverted against correctness labels.')

    symbolic_values = {round(float(example.labels.model_targets.get('symbolic_agreement', 0.0)), 4) for example in artifact.examples}
    completeness_values = {round(float(example.labels.model_targets.get('completeness', 0.0)), 4) for example in artifact.examples}
    step_target_total = sum(len(example.labels.model_targets.get('step_correctness', ())) for example in artifact.examples)
    need(len(symbolic_values) > 2, 'Symbolic agreement supervision collapsed to too few values.')
    need(len(completeness_values) > 2, 'Completeness supervision collapsed to too few values.')
    need(step_target_total > len(artifact.examples), 'Dataset collapsed away step-level correctness supervision.')

    return {
        'split_counts': dict(sorted(split_counts.items())),
        'branch_positive_count': len(branch_positive),
        'branch_negative_count': len(branch_negative),
        'dominant_branch_share': dominant_branch_share,
        'hard_negative_count': hard_negative_count,
        'positive_symbolic_mean': stable_mean(positive_symbolic),
        'negative_symbolic_mean': stable_mean(negative_symbolic),
        'excluded_reason_counts': dict(sorted(Counter(item['exclude_reason'] for item in excluded_rows).items())),
    }


def dataset_rows_from_artifact(artifact):
    rows = []
    for example in artifact.examples:
        targets = dict(example.labels.model_targets)
        legacy = dict(example.labels.legacy_label)
        rows.append(
            {
                'example_id': example.example_id,
                'split': example.split.value,
                'scope': example.scope.value,
                'retention_tag': example.retention_tag.value,
                'problem_id': example.problem_id,
                'branch_id': example.branch_id,
                'prefix_step_count': example.prefix_step_count,
                'total_step_count': example.total_step_count,
                'quality_tag': example.quality_tag.value,
                'verdict': example.verdict.value,
                'problem_text': example.texts.problem_text,
                'prefix_text': example.texts.prefix_text,
                'reasoning_text': example.texts.reasoning_text,
                'answer_text': example.texts.answer_text,
                'canonical_answer': example.texts.canonical_answer,
                'operator_sequence': sj(list(example.texts.operator_sequence)),
                'route_problem_type': sj(list(example.texts.route_problem_type)),
                'route_archetypes': sj(list(example.texts.route_archetypes)),
                'step_correctness': sj(list(targets.get('step_correctness', ()))),
                'logical_consistency': float(targets.get('logical_consistency', 0.0)),
                'symbolic_agreement': float(targets.get('symbolic_agreement', 0.0)),
                'completeness': float(targets.get('completeness', 0.0)),
                'answer_correct_likelihood': float(targets.get('answer_correct_likelihood', 0.0)),
                'repairability': float(targets.get('repairability', 0.0)),
                'verdict_confidence': float(targets.get('verdict_confidence', 0.0)),
                'failure_type': legacy.get('failure_type'),
                'failure_location': legacy.get('failure_location'),
                'hard_negative': bool(example.metadata.get('hard_negative', False)),
                'hard_negative_reason': text(example.metadata.get('hard_negative_reason')),
                'truth_source': text(example.metadata.get('truth_source')),
                'failure_context': sj(example.metadata.get('failure_labels', [])),
                'bundle': sj(example.labels.bundle),
                'provenance': sj(example.provenance.model_dump(mode='json')),
            }
        )
    return rows


def calibration_curve_payload(rows, runtime_manifest, split_name, threshold):
    head = runtime_manifest['heads']['probability']
    y_true = [row.binary_answer_target for row in rows]
    y_prob = [
        train_verifier_script._predict_head(row.features, head['weights'], float(head['bias']))
        for row in rows
    ]
    summary = evaluate_binary_calibration(y_true, y_prob, threshold=threshold, num_buckets=10)
    return {
        'split': split_name,
        'count': summary.count,
        'accuracy': summary.accuracy,
        'mean_confidence': summary.mean_confidence,
        'positive_rate': summary.positive_rate,
        'brier_score': summary.brier_score,
        'expected_calibration_error': summary.expected_calibration_error,
        'max_calibration_error': summary.max_calibration_error,
        'buckets': [
            {
                'bucket_index': bucket.bucket_index,
                'lower_bound': bucket.lower_bound,
                'upper_bound': bucket.upper_bound,
                'count': bucket.count,
                'mean_confidence': bucket.mean_confidence,
                'empirical_accuracy': bucket.empirical_accuracy,
                'confidence_gap': bucket.confidence_gap,
            }
            for bucket in summary.buckets
        ],
    }


In [ ]:
prepare_output_dir(OUT_DIR, OVERWRITE_OUTPUTS)
trace_path = resolve_trace_path()
trace_frame = pd.read_parquet(trace_path)
need(not trace_frame.empty, f'Trace source is empty: {trace_path}')

cases = []
excluded_rows = []
for index, row in enumerate(trace_frame.to_dict(orient='records')):
    case = row_to_trace_case(row, index)
    if case['usable']:
        cases.append(case)
    else:
        excluded_rows.append(case)

need(cases, 'No usable branch traces were available for verifier supervision.')

config = VerifierDistillationConfig(
    include_branch_examples=True,
    include_prefix_examples=True,
    min_prefix_steps=1,
    max_prefix_examples_per_branch=None,
    prefix_stride=1,
    include_terminal_prefix=True,
    retain_plausible_wrong=True,
    retain_unsound_wrong=True,
    retain_symbolic_contradicted=True,
    retain_repaired=True,
    retain_verifier_rejected=True,
)

artifact = build_artifact_from_cases(cases, config)
validation = validate_artifact(artifact, excluded_rows)

dataset_rows = dataset_rows_from_artifact(artifact)
dataset_frame = pd.DataFrame(dataset_rows)
need(not dataset_frame.empty, 'verifier_dataset.parquet would be empty.')
dataset_frame.to_parquet(VERIFIER_DATASET_PATH, index=False)

write_result = write_verifier_training_artifact(artifact, ARTIFACT_ROOT, include_split_files=True)

train_exit = train_verifier_script.main(
    [
        '--dataset-path',
        str(write_result.artifact_dir),
        '--output-dir',
        str(TRAINING_ROOT),
        '--run-name',
        RUN_ID,
        '--seed',
        str(SEED),
        '--expected-artifact-kind',
        ARTIFACT_KIND,
        '--overwrite',
    ]
)
need(train_exit == 0, f'Verifier training pipeline failed with exit code {train_exit}.')

summary_candidates = sorted(TRAINING_ROOT.glob('*/summary.json'), key=lambda path: path.stat().st_mtime, reverse=True)
need(summary_candidates, 'Training summary.json was not produced by the verifier training pipeline.')
training_run_dir = summary_candidates[0].parent
training_summary = json.loads((training_run_dir / 'summary.json').read_text(encoding='utf-8'))
runtime_manifest = json.loads((training_run_dir / 'runtime_model.json').read_text(encoding='utf-8'))
calibration_manifest = json.loads((training_run_dir / 'calibration.json').read_text(encoding='utf-8'))

derived_rows = train_verifier_script._derive_examples(artifact)
rows_by_split = train_verifier_script._group_by_split(derived_rows)
selected_threshold = float(calibration_manifest['selected_probability_threshold'])
calibration_curves = {
    SplitName.VALID.value: calibration_curve_payload(rows_by_split[SplitName.VALID.value], runtime_manifest, SplitName.VALID.value, selected_threshold),
    SplitName.CALIBRATION.value: calibration_curve_payload(rows_by_split[SplitName.CALIBRATION.value], runtime_manifest, SplitName.CALIBRATION.value, selected_threshold),
}
training_metrics = dict(training_summary.get('metrics_by_split', {}))

TRAINING_METRICS_PATH.write_text(json.dumps(training_metrics, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')
CALIBRATION_CURVES_PATH.write_text(json.dumps(calibration_curves, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')

diagnostics = {
    'source_trace_path': str(trace_path),
    'output_dir': str(OUT_DIR),
    'artifact_metadata': artifact.metadata.model_dump(mode='json'),
    'artifact_write_result': write_result.model_dump(mode='json'),
    'trace_record_count': int(len(trace_frame)),
    'usable_branch_trace_count': int(len(cases)),
    'excluded_trace_row_count': int(len(excluded_rows)),
    'excluded_rows_sample': excluded_rows[:20],
    'truth_source_histogram': dict(sorted(Counter(case['truth_source'] for case in cases).items())),
    'failure_context_histogram': dict(sorted(Counter(label for case in cases for label in case['failure_labels']).items())),
    'retention_tag_histogram': dict(sorted(Counter(example.retention_tag.value for example in artifact.examples).items())),
    'hard_negative_reason_histogram': dict(sorted(Counter(text(example.metadata.get('hard_negative_reason')) for example in artifact.examples if example.metadata.get('hard_negative')).items())),
    'validation': validation,
    'training_run_dir': str(training_run_dir),
    'training_summary_path': str(training_run_dir / 'summary.json'),
    'runtime_manifest_path': str(training_run_dir / 'runtime_model.json'),
    'calibration_manifest_path': str(training_run_dir / 'calibration.json'),
}

DIAGNOSTICS_PATH.write_text(json.dumps(diagnostics, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')
diagnostics


In [ ]:
diagnostics = json.loads(DIAGNOSTICS_PATH.read_text(encoding='utf-8'))
training_metrics = json.loads(TRAINING_METRICS_PATH.read_text(encoding='utf-8'))
calibration_curves = json.loads(CALIBRATION_CURVES_PATH.read_text(encoding='utf-8'))
verifier_dataset = pd.read_parquet(VERIFIER_DATASET_PATH)
selected_threshold = json.loads((Path(diagnostics['calibration_manifest_path'])).read_text(encoding='utf-8'))['selected_probability_threshold']

summary = {
    'usable_branch_trace_count': diagnostics['usable_branch_trace_count'],
    'excluded_trace_row_count': diagnostics['excluded_trace_row_count'],
    'branch_balance_dominant_share': diagnostics['validation']['dominant_branch_share'],
    'hard_negative_count': diagnostics['validation']['hard_negative_count'],
    'valid_answer_accuracy': training_metrics.get('valid', {}).get('answer_accuracy_at_threshold'),
    'calibration_answer_accuracy': training_metrics.get('calibration', {}).get('answer_accuracy_at_threshold'),
    'calibration_ece': calibration_curves.get('calibration', {}).get('expected_calibration_error'),
    'selected_probability_threshold': selected_threshold,
}

print(json.dumps(summary, indent=2, ensure_ascii=True, sort_keys=True))
verifier_dataset.head(20)
